# Prototype: Edge Hybrid Attention
DSv4 compression + Linear Attention for O(1) KV cache.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, math, time

## 1. Linear Attention

In [2]:
class LinearAttn(nn.Module):
    def __init__(self, dim, decay=0.99, eps=1e-6):
        super().__init__(); self.decay, self.eps = decay, eps
    def fm(self, x): return F.elu(x) + 1.0
    def forward(self, q, k, v, S=None, z=None):
        qf, kf, vf = self.fm(q), self.fm(k), self.fm(v)
        if S is None:
            S = kf.new_zeros(*q.shape[:2], q.shape[-1], q.shape[-1])
            z = kf.new_zeros(*q.shape[:2], q.shape[-1])
        S = self.decay * S + kf.transpose(-2, -1) @ vf
        z = self.decay * z + kf.sum(-2)
        return (qf @ S) / (qf @ z.unsqueeze(-1)).clamp(min=self.eps), S, z

## 2. KV Compression

In [3]:
class KVCompress(nn.Module):
    def __init__(self, dim, hd, m, overlap=False):
        super().__init__()
        self.m, self.overlap = m, overlap
        self.W_k = nn.Linear(dim, hd, 0); self.W_v = nn.Linear(dim, hd, 0)
        self.W_z = nn.Linear(dim, 1, 0)
        if overlap:
            self.W_kb = nn.Linear(dim, hd, 0); self.W_vb = nn.Linear(dim, hd, 0)
            self.W_zb = nn.Linear(dim, 1, 0)
    def forward(self, h):
        b, n, d = h.shape; m = self.m; nc = n // m; r = nc * m
        hb = h[:, :r].reshape(b, nc, m, d)
        k = self.W_k(hb); v = self.W_v(hb)
        z = self.W_z(hb).squeeze(-1)
        if self.overlap and nc > 1:
            hs = torch.cat([h[:, :1].expand(-1, m, -1), h[:, :r]], 1)[:, :r]
            hs = hs.reshape(b, nc, m, d)
            k = torch.cat([k, self.W_kb(hs)], 2)
            v = torch.cat([v, self.W_vb(hs)], 2)
            z = torch.cat([z, self.W_zb(hs).squeeze(-1)], -1)
        w = F.softmax(z, -1)
        return (w.unsqueeze(-1) * k).sum(2), (w.unsqueeze(-1) * v).sum(2)

## 3-6. CSA, HCA, SWA, Hybrid

In [4]:
class CSA(nn.Module):
    def __init__(self, dim, hd, m=4, tk=64, decay=0.99):
        super().__init__()
        self.tk = tk; self.cp = KVCompress(dim, hd, m, True)
        self.qp = nn.Linear(dim, hd, 0); self.out = nn.Linear(hd, dim, 0)
        self.la = LinearAttn(hd, decay)
    def forward(self, q, h, S=None, z_s=None):
        k, v = self.cp(h); qk = self.qp(q[:, -1:])
        s = (qk @ k.transpose(-2, -1)) / math.sqrt(k.shape[-1])
        _, ix = torch.topk(s, min(self.tk, k.shape[1]), -1)
        ix = ix.unsqueeze(-1).expand(-1, -1, -1, k.shape[-1]).squeeze(1)
        k, v = torch.gather(k, 1, ix), torch.gather(v, 1, ix)
        o, S, z_s = self.la(qk.unsqueeze(1), k.unsqueeze(1), v.unsqueeze(1), S, z_s)
        return self.out(o.squeeze(1)), S, z_s

class HCA(nn.Module):
    def __init__(self, dim, hd, m=128, decay=0.999):
        super().__init__()
        self.cp = KVCompress(dim, hd, m, False); self.out = nn.Linear(hd, dim, 0)
        self.qp = nn.Linear(dim, hd, 0); self.la = LinearAttn(hd, decay)
    def forward(self, q, h, S=None, z_s=None):
        k, v = self.cp(h); qk = self.qp(q[:, -1:]).unsqueeze(1)
        o, S, z_s = self.la(qk, k.unsqueeze(1), v.unsqueeze(1), S, z_s)
        return self.out(o.squeeze(1)), S, z_s

class SWA(nn.Module):
    def __init__(self, dim, nh=4, win=128):
        super().__init__(); self.win=win; self.nh=nh; self.hd=dim//nh
        self.qp=nn.Linear(dim,dim,0); self.kp=nn.Linear(dim,dim,0)
        self.vp=nn.Linear(dim,dim,0); self.op=nn.Linear(dim,dim,0)
    def forward(self, q, h):
        b,n,d=h.shape; c=h[:,-min(n,self.win):]
        k=self.kp(c).view(b,-1,self.nh,self.hd).transpose(1,2)
        v=self.vp(c).view(b,-1,self.nh,self.hd).transpose(1,2)
        q=self.qp(q[:,-1:]).view(b,1,self.nh,self.hd).transpose(1,2)
        a=F.softmax((q@k.transpose(-2,-1))/math.sqrt(self.hd),-1)
        return self.op((a@v).transpose(1,2).reshape(b,1,d).squeeze(1))

class HybridBlock(nn.Module):
    def __init__(self, dim=256, hd=64, nh=4, cm=4, ck=64, cd=0.99, hm=128, hd2=0.999, sw=128):
        super().__init__()
        self.csa = CSA(dim, hd, cm, ck, cd)
        self.hca = HCA(dim, hd, hm, hd2)
        self.swa = SWA(dim, nh, sw)
        self.g = nn.Parameter(torch.ones(3))
    def forward(self, q, h, cs=None, cz=None, hs=None, hz=None):
        co,cs,cz = self.csa(q,h,cs,cz)
        ho,hs,hz = self.hca(q,h,hs,hz)
        so = self.swa(q,h)
        g = F.softmax(self.g,0)
        return g[0]*co+g[1]*ho+g[2]*so, (cs,cz), (hs,hz)

## 7. Simulation & Results

In [5]:
def sim(m, nf=1000, dim=256, ch=128):
    cs=cz=hs=hz=None
    for i in range(0,nf,ch):
        h=torch.randn(1,ch,dim); q=torch.randn(1,1,dim)
        _,(cs,cz),(hs,hz)=m(q,h,cs,cz,hs,hz)
    return cs,hs

m = HybridBlock(256)
print(f"Params: {sum(p.numel() for p in m.parameters()):,}")
cs, hs = sim(m, 1000)
print(f"CSA state: {list(cs.shape)} = {cs.element_size()*cs.numel()/1024:.1f} KB")
print(f"HCA state: {list(hs.shape)} = {hs.element_size()*hs.numel()/1024:.1f} KB")
total=(cs.element_size()*cs.numel()+hs.element_size()*hs.numel())/1024
print(f"Total KV cache: {total:.1f} KB (O(1), independent of frames)")

Params: 426,755
CSA state: [1, 1, 64, 64] = 16.0 KB
HCA state: [1, 1, 64, 64] = 16.0 KB
Total KV cache: 32.0 KB (O(1), independent of frames)


In [6]:
dim=256
print(f"{'Frames':>10} | {'MHA(MB)':>10} | {'Ours(KB)':>10} | {'Time(s)':>10}")
print('-'*44)
for n in [100,1000,10000]:
    mha=n*dim*4*2/1024**2; t0=time.time()
    cs,hs=sim(m,n)
    okb=(cs.element_size()*cs.numel()+hs.element_size()*hs.numel())/1024
    print(f"{n:>10} | {mha:>8.2f} MB | {okb:>8.1f} KB | {time.time()-t0:>8.3f}s")

    Frames |    MHA(MB) |   Ours(KB) |    Time(s)
--------------------------------------------
       100 |     0.20 MB |     32.0 KB |    0.008s
      1000 |     1.95 MB |     32.0 KB |    0.041s


     10000 |    19.53 MB |     32.0 KB |    0.448s


**KV cache is O(1)**: 10,000 frames = 19.53 MB for MHA vs 32 KB for our approach.